**Install Libraries**

In [ ]:
!pip install yfinance beautifulsoup4 requests pandas -q

**The Scraper**

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import warnings
warnings.filterwarnings("ignore")

# Scrape Nifty 50 list from Wikipedia
print(" Scraping Nifty 50 company list from Wikipedia...")

url = "https://en.wikipedia.org/wiki/NIFTY_50"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

companies = []
tables = soup.find_all("table", class_="wikitable")

for table in tables:
    rows = table.find_all("tr")
    for row in rows[1:]:
        cols = row.find_all("td")
        if len(cols) >= 2:
            name = cols[0].text.strip()
            symbol = cols[1].text.strip()
            sector = cols[2].text.strip() if len(cols) > 2 else "Unknown"
            if symbol and name:
                companies.append({
                    "Company": name,
                    "Symbol": symbol + ".NS",   # Yahoo Finance format
                    "Sector": sector
                })

df_companies = pd.DataFrame(companies).drop_duplicates().head(25)
print(f" Found {len(df_companies)} companies\n")
print(df_companies.to_string(index=False))

 Scraping Nifty 50 company list from Wikipedia...
 Found 25 companies

                 Company        Symbol                         Sector
       Adani Enterprises   ADANIENT.NS                Metals & Mining
       Adani Ports & SEZ ADANIPORTS.NS                       Services
        Apollo Hospitals APOLLOHOSP.NS                     Healthcare
            Asian Paints ASIANPAINT.NS              Consumer Durables
               Axis Bank   AXISBANK.NS             Financial Services
              Bajaj Auto BAJAJ-AUTO.NS Automobile and Auto Components
           Bajaj Finance BAJFINANCE.NS             Financial Services
           Bajaj Finserv BAJAJFINSV.NS             Financial Services
      Bharat Electronics        BEL.NS                  Capital Goods
           Bharti Airtel BHARTIARTL.NS              Telecommunication
                   Cipla      CIPLA.NS                     Healthcare
              Coal India  COALINDIA.NS    Oil, Gas & Consumable Fuels
Dr. Reddy's Laborat

**yfinance Financial Data Puller**

In [ ]:

# yfinance
print("\n Pulling financial data from Yahoo Finance")

records = []

for _, row in df_companies.iterrows():
    company = row["Company"]
    symbol  = row["Symbol"]
    sector  = row["Sector"]

    try:
        ticker = yf.Ticker(symbol)
        info   = ticker.info

        record = {
            "Company"          : company,
            "Symbol"           : symbol,
            "Sector"           : sector,
            # Valuation
            "Market_Cap_Cr"    : round(info.get("marketCap", 0) / 1e7, 2),
            "PE_Ratio"         : info.get("trailingPE", None),
            "PB_Ratio"         : info.get("priceToBook", None),
            "EV_EBITDA"        : info.get("enterpriseToEbitda", None),
            # Profitability
            "Revenue_Cr"       : round(info.get("totalRevenue", 0) / 1e7, 2),
            "Net_Profit_Margin": round(info.get("profitMargins", 0) * 100, 2),
            "EBITDA_Margin"    : round(info.get("ebitdaMargins", 0) * 100, 2) if info.get("ebitdaMargins") else None,
            "ROE"              : round(info.get("returnOnEquity", 0) * 100, 2),
            "ROA"              : round(info.get("returnOnAssets", 0) * 100, 2),
            # Leverage
            "Debt_to_Equity"   : info.get("debtToEquity", None),
            "Current_Ratio"    : info.get("currentRatio", None),
            # Per Share
            "EPS"              : info.get("trailingEps", None),
            "Book_Value"       : info.get("bookValue", None),
            # Growth
            "Revenue_Growth_YoY": round(info.get("revenueGrowth", 0) * 100, 2),
            "Earnings_Growth_YoY": round(info.get("earningsGrowth", 0) * 100, 2) if info.get("earningsGrowth") else None,
        }

        records.append(record)
        print(f"   {company} ({symbol})")

    except Exception as e:
        print(f"  {company} ({symbol}) — {e}")
        records.append({"Company": company, "Symbol": symbol, "Sector": sector})

    time.sleep(1)

df_financials = pd.DataFrame(records)
print(f"\n Done! Shape: {df_financials.shape}")
print(df_financials.head())


 Pulling financial data from Yahoo Finance
   Adani Enterprises (ADANIENT.NS)
   Adani Ports & SEZ (ADANIPORTS.NS)
   Apollo Hospitals (APOLLOHOSP.NS)
   Asian Paints (ASIANPAINT.NS)
   Axis Bank (AXISBANK.NS)
   Bajaj Auto (BAJAJ-AUTO.NS)
   Bajaj Finance (BAJFINANCE.NS)
   Bajaj Finserv (BAJAJFINSV.NS)
   Bharat Electronics (BEL.NS)
   Bharti Airtel (BHARTIARTL.NS)
   Cipla (CIPLA.NS)
   Coal India (COALINDIA.NS)
   Dr. Reddy's Laboratories (DRREDDY.NS)
   Eicher Motors (EICHERMOT.NS)
   Eternal (ETERNAL.NS)
   Grasim Industries (GRASIM.NS)
   HCLTech (HCLTECH.NS)
   HDFC Bank (HDFCBANK.NS)
   HDFC Life (HDFCLIFE.NS)
   Hindalco Industries (HINDALCO.NS)
   Hindustan Unilever (HINDUNILVR.NS)
   ICICI Bank (ICICIBANK.NS)
   IndiGo (INDIGO.NS)
   Infosys (INFY.NS)
   ITC (ITC.NS)

 Done! Shape: (25, 18)
             Company         Symbol              Sector  Market_Cap_Cr  \
0  Adani Enterprises    ADANIENT.NS     Metals & Mining      256329.46   
1  Adani Ports & SEZ  ADANIPORTS.NS  

**Save Raw Data**

In [ ]:
df_financials.to_csv("raw_financial_data.csv", index=False)
print(" Saved -> raw_financial_data.csv")
print(f"\nColumns: {list(df_financials.columns)}")
print(f"\nMissing values:\n{df_financials.isnull().sum()}")

 Saved -> raw_financial_data.csv

Columns: ['Company', 'Symbol', 'Sector', 'Market_Cap_Cr', 'PE_Ratio', 'PB_Ratio', 'EV_EBITDA', 'Revenue_Cr', 'Net_Profit_Margin', 'EBITDA_Margin', 'ROE', 'ROA', 'Debt_to_Equity', 'Current_Ratio', 'EPS', 'Book_Value', 'Revenue_Growth_YoY', 'Earnings_Growth_YoY']

Missing values:
Company                 0
Symbol                  0
Sector                  0
Market_Cap_Cr           0
PE_Ratio                0
PB_Ratio                0
EV_EBITDA               4
Revenue_Cr              0
Net_Profit_Margin       0
EBITDA_Margin           4
ROE                     0
ROA                     0
Debt_to_Equity          3
Current_Ratio          20
EPS                     0
Book_Value              0
Revenue_Growth_YoY      0
Earnings_Growth_YoY     1
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("raw_financial_data.csv")
print(f"Loaded: {df.shape}")

Loaded: (25, 18)


**Cleaning**

In [ ]:
# Fill nulls intelligently
df["Current_Ratio"]     = df["Current_Ratio"].fillna(df["Current_Ratio"].median())
df["EV_EBITDA"]         = df["EV_EBITDA"].fillna(df["EV_EBITDA"].median())
df["EBITDA_Margin"]     = df["EBITDA_Margin"].fillna(df["Net_Profit_Margin"] * 1.3)  # Proxy
df["Debt_to_Equity"]    = df["Debt_to_Equity"].fillna(0)  # No debt reported = 0
df["Earnings_Growth_YoY"] = df["Earnings_Growth_YoY"].fillna(0)

# Remove any completely empty rows
df = df.dropna(subset=["Company", "Revenue_Cr", "ROE"])

print(f"After cleaning: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

After cleaning: (25, 18)
Missing values:
Company                0
Symbol                 0
Sector                 0
Market_Cap_Cr          0
PE_Ratio               0
PB_Ratio               0
EV_EBITDA              0
Revenue_Cr             0
Net_Profit_Margin      0
EBITDA_Margin          0
ROE                    0
ROA                    0
Debt_to_Equity         0
Current_Ratio          0
EPS                    0
Book_Value             0
Revenue_Growth_YoY     0
Earnings_Growth_YoY    0
dtype: int64


**Additional Ratio Calculations**

In [ ]:
# Price-to-Earnings-to-Growth (PEG Ratio)
df["PEG_Ratio"] = df.apply(
    lambda r: round(r["PE_Ratio"] / r["Earnings_Growth_YoY"], 2)
    if r["Earnings_Growth_YoY"] and r["Earnings_Growth_YoY"] > 0 else None, axis=1
)

# Earnings Quality Score (higher margin + higher ROE = better)
df["Earnings_Quality"] = round((df["Net_Profit_Margin"] + df["ROE"]) / 2, 2)

# Leverage Risk Flag
df["Leverage_Risk"] = df["Debt_to_Equity"].apply(
    lambda x: "High" if x > 150 else ("Medium" if x > 75 else "Low")
)

# Valuation Category
df["Valuation_Tag"] = df["PE_Ratio"].apply(
    lambda x: "Overvalued" if x > 40 else ("Fairly Valued" if x > 15 else "Undervalued")
)

# Growth Category
df["Growth_Tag"] = df["Revenue_Growth_YoY"].apply(
    lambda x: "High Growth" if x > 15 else ("Moderate Growth" if x > 5 else "Low Growth")
)


**Financial Health Score (0 to 100)**

In [ ]:
def health_score(row):
    score = 0

    # Profitability (30 pts)
    if row["Net_Profit_Margin"] > 20: score += 30
    elif row["Net_Profit_Margin"] > 10: score += 20
    elif row["Net_Profit_Margin"] > 0:  score += 10

    # Return on Equity (20 pts)
    if row["ROE"] > 20: score += 20
    elif row["ROE"] > 10: score += 12
    elif row["ROE"] > 0:  score += 5

    # Leverage (20 pts)
    if row["Debt_to_Equity"] < 30:   score += 20
    elif row["Debt_to_Equity"] < 75:  score += 12
    elif row["Debt_to_Equity"] < 150: score += 5

    # Valuation (15 pts)
    if row["PE_Ratio"] < 20:   score += 15
    elif row["PE_Ratio"] < 35: score += 8
    else:                       score += 3

    # Growth (15 pts)
    if row["Revenue_Growth_YoY"] > 15:   score += 15
    elif row["Revenue_Growth_YoY"] > 5:  score += 10
    elif row["Revenue_Growth_YoY"] >= 0: score += 5

    return score

df["Health_Score"] = df.apply(health_score, axis=1)

# Health Rating
df["Health_Rating"] = df["Health_Score"].apply(
    lambda x: " Strong" if x >= 70 else (" Moderate" if x >= 45 else " Weak")
)

**Rank Companies**

In [ ]:
df["Health_Rank"] = df["Health_Score"].rank(ascending=False).astype(int)
df = df.sort_values("Health_Rank")

**Save**

In [ ]:
df.to_csv("clean_financial_data.csv", index=False)
print("\n Saved → clean_financial_data.csv")

# Preview
print("\n Top 10 Healthiest Companies:")
print(df[["Company", "Sector", "Health_Score", "Health_Rating", "Health_Rank",
          "PE_Ratio", "ROE", "Net_Profit_Margin", "Debt_to_Equity"]].head(10).to_string(index=False))


 Saved → clean_financial_data.csv

 Top 10 Healthiest Companies:
                 Company                         Sector  Health_Score Health_Rating  Health_Rank  PE_Ratio   ROE  Net_Profit_Margin  Debt_to_Equity
               HDFC Bank             Financial Services            92        Strong            1 17.220610 14.02              26.19           0.000
               Axis Bank             Financial Services            87        Strong            2 14.845030 13.63              34.17           0.000
              ICICI Bank             Financial Services            82        Strong            3 17.017351 16.80              27.51           0.000
                 Infosys         Information Technology            80        Strong            4 18.620882 32.68              16.15          10.531
                 HCLTech         Information Technology            78        Strong            5 23.741764 23.19              13.02           9.747
                     ITC     Fast Moving Consu

**DuckDB SQL Analysis**

In [ ]:
!pip install duckdb -q

import duckdb
import pandas as pd

df = pd.read_csv("clean_financial_data.csv")

con = duckdb.connect()
con.register("financials", df)

print(" DuckDB connected — running 10 business queries...\n")

# Top 5 Companies by Financial Health Score
q1 = con.execute("""
    SELECT Company, Sector, Health_Score, Health_Rating
    FROM financials
    ORDER BY Health_Score DESC
    LIMIT 5
""").df()
print(" Top 5 Financially Healthiest Companies:")
print(q1.to_string(index=False))


# Sector-wise Average Financial Health Score
q2 = con.execute("""
    SELECT
        Sector,
        ROUND(AVG(Health_Score), 1)         AS Avg_Health_Score,
        ROUND(AVG(ROE), 1)                  AS Avg_ROE,
        ROUND(AVG(Net_Profit_Margin), 1)    AS Avg_Net_Margin,
        COUNT(*)                            AS No_of_Companies
    FROM financials
    GROUP BY Sector
    ORDER BY Avg_Health_Score DESC
""").df()
print("\n Sector-wise Financial Health Benchmarks:")
print(q2.to_string(index=False))

# Overleveraged Companies (High Debt Risk)

q3 = con.execute("""
    SELECT Company, Sector, Debt_to_Equity, Leverage_Risk, Health_Score
    FROM financials
    WHERE Leverage_Risk = 'High'
    ORDER BY Debt_to_Equity DESC
""").df()
print("\n High Leverage Risk Companies (D/E > 150):")
print(q3.to_string(index=False))

# Undervalued AND Financially Strong

q4 = con.execute("""
    SELECT Company, Sector, PE_Ratio, Health_Score, ROE, Net_Profit_Margin
    FROM financials
    WHERE Valuation_Tag = 'Undervalued'
      AND Health_Score >= 60
    ORDER BY Health_Score DESC
""").df()
print("\n Hidden Gems (Undervalued + Strong Fundamentals):")
print(q4.to_string(index=False))

# Window Function: Health Score Rank Within Each Sector
q5 = con.execute("""
    SELECT
        Company,
        Sector,
        Health_Score,
        RANK() OVER (PARTITION BY Sector ORDER BY Health_Score DESC) AS Sector_Rank
    FROM financials
    ORDER BY Sector, Sector_Rank
""").df()
print("\n Health Score Rank Within Each Sector:")
print(q5.to_string(index=False))

# Profitability Leaders (Top ROE + Net Margin)
q6 = con.execute("""
    SELECT
        Company,
        Sector,
        ROE,
        Net_Profit_Margin,
        ROUND((ROE + Net_Profit_Margin) / 2, 1) AS Profitability_Score
    FROM financials
    ORDER BY Profitability_Score DESC
    LIMIT 8
""").df()
print("\n Profitability Leaders (ROE + Net Margin):")
print(q6.to_string(index=False))

# Revenue Growth Leaders vs Laggards
q7 = con.execute("""
    SELECT
        Company,
        Sector,
        Revenue_Growth_YoY,
        Earnings_Growth_YoY,
        Growth_Tag
    FROM financials
    ORDER BY Revenue_Growth_YoY DESC
""").df()
print("\n Revenue Growth Leaders vs Laggards:")
print(q7.to_string(index=False))

# Valuation vs Health Score (Overvalued but Weak?)
q8 = con.execute("""
    SELECT
        Company,
        Sector,
        PE_Ratio,
        Valuation_Tag,
        Health_Score,
        Health_Rating,
        CASE
            WHEN Valuation_Tag = 'Overvalued' AND Health_Score < 50
                THEN ' Avoid — Overvalued & Weak'
            WHEN Valuation_Tag = 'Overvalued' AND Health_Score >= 70
                THEN ' Premium — Overvalued but Strong'
            WHEN Valuation_Tag = 'Undervalued' AND Health_Score >= 60
                THEN ' Buy Signal'
            ELSE ' Watch'
        END AS Investment_Signal
    FROM financials
    ORDER BY PE_Ratio DESC
""").df()
print("\n Investment Signal Matrix:")
print(q8.to_string(index=False))

#  Bottom 5 Red Flag Companies
q9 = con.execute("""
    SELECT
        Company,
        Sector,
        Health_Score,
        Health_Rating,
        Debt_to_Equity,
        Net_Profit_Margin,
        ROE
    FROM financials
    ORDER BY Health_Score ASC
    LIMIT 5
""").df()
print("\n Bottom 5 Red Flag Companies:")
print(q9.to_string(index=False))

# Executive Summary: Portfolio Snapshot
q10 = con.execute("""
    SELECT
        COUNT(*)                                            AS Total_Companies,
        ROUND(AVG(Health_Score), 1)                        AS Avg_Health_Score,
        ROUND(AVG(PE_Ratio), 1)                            AS Avg_PE,
        ROUND(AVG(ROE), 1)                                 AS Avg_ROE,
        ROUND(AVG(Net_Profit_Margin), 1)                   AS Avg_Net_Margin,
        COUNT(CASE WHEN Health_Rating LIKE '%Strong%' THEN 1 END)   AS Strong_Companies,
        COUNT(CASE WHEN Health_Rating LIKE '%Moderate%' THEN 1 END) AS Moderate_Companies,
        COUNT(CASE WHEN Health_Rating LIKE '%Weak%' THEN 1 END)     AS Weak_Companies,
        COUNT(CASE WHEN Leverage_Risk = 'High' THEN 1 END)          AS High_Leverage_Count
    FROM financials
""").df()
print("\n Executive Portfolio Snapshot:")
print(q10.to_string(index=False))

# Save all query results
with pd.ExcelWriter("sql_analysis_results.xlsx", engine="openpyxl") as writer:
    q1.to_excel(writer, sheet_name="Q1_Top5_Health",       index=False)
    q2.to_excel(writer, sheet_name="Q2_Sector_Benchmarks", index=False)
    q3.to_excel(writer, sheet_name="Q3_High_Leverage",     index=False)
    q4.to_excel(writer, sheet_name="Q4_Hidden_Gems",       index=False)
    q5.to_excel(writer, sheet_name="Q5_Sector_Ranks",      index=False)
    q6.to_excel(writer, sheet_name="Q6_Profitability",     index=False)
    q7.to_excel(writer, sheet_name="Q7_Growth",            index=False)
    q8.to_excel(writer, sheet_name="Q8_Investment_Signal", index=False)
    q9.to_excel(writer, sheet_name="Q9_Red_Flags",         index=False)
    q10.to_excel(writer, sheet_name="Q10_Exec_Summary",    index=False)

print("\n All 10 queries saved → sql_analysis_results.xlsx")

 DuckDB connected — running 10 business queries...

 Top 5 Financially Healthiest Companies:
   Company                 Sector  Health_Score Health_Rating
 HDFC Bank     Financial Services            92        Strong
 Axis Bank     Financial Services            87        Strong
ICICI Bank     Financial Services            82        Strong
   Infosys Information Technology            80        Strong
   HCLTech Information Technology            78        Strong

 Sector-wise Financial Health Benchmarks:
                        Sector  Avg_Health_Score  Avg_ROE  Avg_Net_Margin  No_of_Companies
        Information Technology              79.0     27.9            14.6                2
                 Capital Goods              68.0      0.0            22.5                1
            Financial Services              67.0      9.3            23.3                6
    Fast Moving Consumer Goods              66.5      0.0            33.1                2
   Oil, Gas & Consumable Fuels       

**Excel Financial Model**

In [ ]:
!pip install openpyxl -q

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("clean_financial_data.csv")

wb = Workbook()
wb.remove(wb.active)

def fill(color):
    return PatternFill("solid", fgColor=color)

def fnt(bold=False, size=10, color="000000"):
    return Font(bold=bold, size=size, color=color)

def aln(h="center"):
    return Alignment(horizontal=h, vertical="center", wrap_text=True)

def bdr():
    s = Side(style="thin", color="CCCCCC")
    return Border(left=s, right=s, top=s, bottom=s)

def write(ws, row, col, value, bg="FFFFFF", bold=False, size=10, color="000000", h="center"):
    c = ws.cell(row=row, column=col, value=value)
    c.fill = fill(bg)
    c.font = fnt(bold=bold, size=size, color=color)
    c.alignment = aln(h)
    c.border = bdr()
    return c


ws1 = wb.create_sheet("Executive Summary")
ws1.sheet_view.showGridLines = False
for col, w in zip(["A","B","C","D","E"], [32,20,20,20,20]):
    ws1.column_dimensions[col].width = w

write(ws1, 1, 1, "NIFTY 50 - Financial Due Diligence Report", bg="1F3864", bold=True, size=15, color="FFFFFF")
ws1.merge_cells("A1:E1")
ws1.row_dimensions[1].height = 45

write(ws1, 2, 1, "Corporate Financial Health Monitor | Ojaswi | April 2026", bg="2E75B6", size=11, color="FFFFFF")
ws1.merge_cells("A2:E2")
ws1.row_dimensions[2].height = 22

write(ws1, 4, 1, "Key Performance Indicators", bg="2E75B6", bold=True, size=12, color="FFFFFF")
ws1.merge_cells("A4:E4")
ws1.row_dimensions[4].height = 26

kpis = [
    ("Total Companies",    len(df)),
    ("Avg Health Score",   f"{df['Health_Score'].mean():.1f} / 100"),
    ("Strong Companies",   len(df[df['Health_Score'] >= 70])),
    ("High Leverage Risk", len(df[df['Leverage_Risk'] == 'High'])),
    ("Avg P/E Ratio",      f"{df['PE_Ratio'].mean():.1f}x"),
    ("Avg ROE",            f"{df['ROE'].mean():.1f}%"),
    ("Avg Net Margin",     f"{df['Net_Profit_Margin'].mean():.1f}%"),
    ("Undervalued Cos",    len(df[df['Valuation_Tag'] == 'Undervalued'])),
]

kpi_bg    = ["FDEBD0","D5F5E3","D6EAF8","FDEDEC","FDEBD0","D5F5E3","D6EAF8","FDEDEC"]
kpi_color = ["784212","1E8449","1A5276","922B21","784212","1E8449","1A5276","922B21"]

# label and value written in separate rows to avoid merged cell write conflict
for i, (label, value) in enumerate(kpis):
    col       = (i % 4) + 1
    label_row = 6 if i < 4 else 9
    value_row = label_row + 1
    write(ws1, label_row, col, label, bg=kpi_bg[i], bold=True, size=9,  color=kpi_color[i])
    write(ws1, value_row, col, value, bg=kpi_bg[i], bold=True, size=14, color=kpi_color[i])
    ws1.row_dimensions[label_row].height = 20
    ws1.row_dimensions[value_row].height = 32

write(ws1, 12, 1, "Key Findings and Recommendations", bg="1F3864", bold=True, size=12, color="FFFFFF")
ws1.merge_cells("A12:E12")
ws1.row_dimensions[12].height = 28

top3   = df.nlargest(3, "Health_Score")["Company"].tolist()
bottom3= df.nsmallest(3, "Health_Score")["Company"].tolist()
hidden = df[(df["Valuation_Tag"] == "Undervalued") & (df["Health_Score"] >= 60)]["Company"].tolist()

findings = [
    ("Top Performers",       f"{', '.join(top3)} lead with the highest financial health scores across all sectors."),
    ("High Risk Companies",  f"{', '.join(bottom3)} show weak fundamentals and are flagged for further review."),
    ("Hidden Gems",          f"{', '.join(hidden) if hidden else 'None identified'} are undervalued with strong underlying fundamentals."),
    ("Recommendation",       "Prioritize companies rated Strong with low leverage and high growth for investment screening."),
]

for i, (title, text) in enumerate(findings):
    r = 13 + i
    write(ws1, r, 1, title, bg="D6EAF8", bold=True, size=10, color="1F3864", h="left")
    for col in range(2, 6):
        write(ws1, r, col, text if col == 2 else "", bg="F8F9FA", size=10, color="000000", h="left")
    ws1.merge_cells(f"B{r}:E{r}")
    ws1.row_dimensions[r].height = 28


ws2 = wb.create_sheet("Ratio Dashboard")
ws2.sheet_view.showGridLines = False

cols_to_show = ["Company","Sector","Market_Cap_Cr","PE_Ratio","PB_Ratio",
                "ROE","ROA","Net_Profit_Margin","Debt_to_Equity",
                "Current_Ratio","Revenue_Growth_YoY","Health_Score",
                "Health_Rating","Valuation_Tag","Leverage_Risk","Growth_Tag"]

widths = [28,26,16,12,12,12,12,18,16,14,20,14,16,16,14,14]
for i, w in enumerate(widths, 1):
    ws2.column_dimensions[get_column_letter(i)].width = w

write(ws2, 1, 1, "Financial Ratio Dashboard - All Companies", bg="1F3864", bold=True, size=14, color="FFFFFF")
ws2.merge_cells(f"A1:{get_column_letter(len(cols_to_show))}1")
ws2.row_dimensions[1].height = 35

for j, col in enumerate(cols_to_show, 1):
    write(ws2, 2, j, col.replace("_"," "), bg="2E75B6", bold=True, size=10, color="FFFFFF")
ws2.row_dimensions[2].height = 28

for i, (_, row) in enumerate(df[cols_to_show].iterrows()):
    r  = i + 3
    bg = "FFFFFF" if i % 2 == 0 else "F0F6FF"
    for j, val in enumerate(row, 1):
        write(ws2, r, j, val, bg=bg, size=10)
    ws2.row_dimensions[r].height = 22

# color scale on health score column to show gradient from weak to strong
ws2.conditional_formatting.add(
    f"L3:L{2+len(df)}",
    ColorScaleRule(start_type="min", start_color="FF6B6B",
                   mid_type="percentile", mid_value=50, mid_color="FFD93D",
                   end_type="max", end_color="6BCB77")
)


ws3 = wb.create_sheet("Red Flag Monitor")
ws3.sheet_view.showGridLines = False

red_cols = ["Company","Sector","Health_Score","Health_Rating",
            "Debt_to_Equity","Net_Profit_Margin","Leverage_Risk"]
for i, w in enumerate([28,26,14,16,16,20,14], 1):
    ws3.column_dimensions[get_column_letter(i)].width = w

write(ws3, 1, 1, "Red Flag Monitor - High Risk Companies", bg="922B21", bold=True, size=14, color="FFFFFF")
ws3.merge_cells(f"A1:{get_column_letter(len(red_cols))}1")
ws3.row_dimensions[1].height = 35

for j, col in enumerate(red_cols, 1):
    write(ws3, 2, j, col.replace("_"," "), bg="C0392B", bold=True, size=10, color="FFFFFF")
ws3.row_dimensions[2].height = 28

df_red = df.sort_values("Health_Score")
for i, (_, row) in enumerate(df_red[red_cols].iterrows()):
    r     = i + 3
    score = df_red.iloc[i]["Health_Score"]
    # row color based on risk level - red weak, yellow moderate, green strong
    bg    = "FADBD8" if score < 45 else ("FEF9E7" if score < 70 else "EAFAF1")
    for j, val in enumerate(row, 1):
        write(ws3, r, j, val, bg=bg, size=10)
    ws3.row_dimensions[r].height = 22


ws4 = wb.create_sheet("Peer Benchmarking")
ws4.sheet_view.showGridLines = False

bench_cols = ["Sector","Avg Health Score","Avg ROE","Avg Net Margin","Avg PE","Companies"]
for i, w in enumerate([28,20,16,18,14,14], 1):
    ws4.column_dimensions[get_column_letter(i)].width = w

write(ws4, 1, 1, "Sector-wise Peer Benchmarking", bg="1F3864", bold=True, size=14, color="FFFFFF")
ws4.merge_cells(f"A1:{get_column_letter(len(bench_cols))}1")
ws4.row_dimensions[1].height = 35

for j, col in enumerate(bench_cols, 1):
    write(ws4, 2, j, col, bg="2E75B6", bold=True, size=10, color="FFFFFF")
ws4.row_dimensions[2].height = 28

sector_bench = df.groupby("Sector").agg(
    Avg_Health_Score   = ("Health_Score",      "mean"),
    Avg_ROE            = ("ROE",               "mean"),
    Avg_Net_Margin     = ("Net_Profit_Margin",  "mean"),
    Avg_PE             = ("PE_Ratio",           "mean"),
    Companies          = ("Company",            "count")
).round(1).reset_index().sort_values("Avg_Health_Score", ascending=False)

for i, (_, row) in enumerate(sector_bench.iterrows()):
    r  = i + 3
    bg = "FFFFFF" if i % 2 == 0 else "EBF5FB"
    for j, val in enumerate(row, 1):
        write(ws4, r, j, val, bg=bg, size=10)
    ws4.row_dimensions[r].height = 22

ws4.conditional_formatting.add(
    f"B3:B{2+len(sector_bench)}",
    ColorScaleRule(start_type="min", start_color="FF6B6B",
                   end_type="max",   end_color="6BCB77")
)


ws5 = wb.create_sheet("DCF Valuation Model")
ws5.sheet_view.showGridLines = False
for col, w in zip(["A","B","C","D","E","F"], [28,18,18,18,18,18]):
    ws5.column_dimensions[col].width = w

write(ws5, 1, 1, "DCF Valuation Model - Reliance Industries", bg="1F3864", bold=True, size=14, color="FFFFFF")
ws5.merge_cells("A1:F1")
ws5.row_dimensions[1].height = 40

write(ws5, 2, 1, "Discounted Cash Flow Analysis | 5-Year Projection | April 2026", bg="2E75B6", size=11, color="FFFFFF")
ws5.merge_cells("A2:F2")
ws5.row_dimensions[2].height = 22

rel          = df[df["Company"].str.contains("Reliance", na=False)]
base_revenue = float(rel["Revenue_Cr"].values[0]) if len(rel) > 0 else 900000
base_margin  = float(rel["Net_Profit_Margin"].values[0]) if len(rel) > 0 else 8.0

assumptions = [
    ("Assumptions",          "Base",                      "Bull",                        "Bear"),
    ("Base Revenue (Cr)",    f"{base_revenue:,.0f}",      f"{base_revenue:,.0f}",        f"{base_revenue:,.0f}"),
    ("Revenue Growth Rate",  "8%",                        "12%",                         "4%"),
    ("Net Profit Margin",    f"{base_margin:.1f}%",       f"{base_margin+2:.1f}%",       f"{base_margin-2:.1f}%"),
    ("WACC",                 "10%",                       "10%",                         "10%"),
    ("Terminal Growth Rate", "4%",                        "4%",                          "4%"),
]

for i, row_data in enumerate(assumptions):
    r  = i + 4
    bg = "1F3864" if i == 0 else ("F0F6FF" if i % 2 == 0 else "FFFFFF")
    fc = "FFFFFF" if i == 0 else "000000"
    for j, val in enumerate(row_data, 1):
        write(ws5, r, j, val, bg=bg, bold=(i==0), size=10, color=fc)
    ws5.row_dimensions[r].height = 24

write(ws5, 11, 1, "5-Year Free Cash Flow Projection (Base Case)", bg="2E75B6", bold=True, size=12, color="FFFFFF")
ws5.merge_cells("A11:F11")
ws5.row_dimensions[11].height = 28

for j, y in enumerate(["Metric","FY2026","FY2027","FY2028","FY2029","FY2030"], 1):
    write(ws5, 12, j, y, bg="1F3864", bold=True, size=10, color="FFFFFF")
ws5.row_dimensions[12].height = 26

growth   = 0.08
margin   = base_margin / 100
wacc     = 0.10
revenues, profits, fcfs, pv_fcfs = [], [], [], []
rev = base_revenue

for yr in range(1, 6):
    rev    = rev * (1 + growth)
    profit = rev * margin
    fcf    = profit * 0.75
    # discount each year's fcf back to present value
    pv     = fcf / ((1 + wacc) ** yr)
    revenues.append(round(rev, 0))
    profits.append(round(profit, 0))
    fcfs.append(round(fcf, 0))
    pv_fcfs.append(round(pv, 0))

proj_rows = [
    ("Revenue (Cr)",          revenues),
    ("Net Profit (Cr)",       profits),
    ("Free Cash Flow (Cr)",   fcfs),
    ("PV of FCF (Cr)",        pv_fcfs),
]

for i, (label, values) in enumerate(proj_rows):
    r  = 13 + i
    bg = "F0F6FF" if i % 2 == 0 else "FFFFFF"
    write(ws5, r, 1, label, bg="D6EAF8", bold=True, size=10, color="1F3864", h="left")
    for j, v in enumerate(values, 2):
        c = write(ws5, r, j, v, bg=bg, size=10)
        c.number_format = "#,##0"
    ws5.row_dimensions[r].height = 22

terminal_value = (fcfs[-1] * 1.04) / (wacc - 0.04)
pv_terminal    = round(terminal_value / ((1 + wacc) ** 5), 0)
enterprise_val = round(sum(pv_fcfs) + pv_terminal, 0)

write(ws5, 18, 1, "Valuation Summary", bg="1F3864", bold=True, size=12, color="FFFFFF")
ws5.merge_cells("A18:F18")
ws5.row_dimensions[18].height = 28

val_items = [
    ("Sum of PV (FCFs)",     f"{sum(pv_fcfs):,.0f} Cr"),
    ("PV of Terminal Value", f"{pv_terminal:,.0f} Cr"),
    ("Enterprise Value",     f"{enterprise_val:,.0f} Cr"),
]

for i, (k, v) in enumerate(val_items):
    r  = 19 + i
    bg = "E8F4FD" if i % 2 == 0 else "FFFFFF"
    write(ws5, r, 1, k, bg=bg, bold=True, size=10, color="1F3864", h="left")
    ws5.merge_cells(f"B{r}:F{r}")
    write(ws5, r, 2, v, bg=bg, bold=True, size=12, color="1A5276")
    ws5.row_dimensions[r].height = 26


output_path = "Financial_Due_Diligence_Report.xlsx"
wb.save(output_path)
print(f"Saved -> {output_path}")
for s in wb.sheetnames:
    print(f"  {s}")

Saved -> Financial_Due_Diligence_Report.xlsx
  Executive Summary
  Ratio Dashboard
  Red Flag Monitor
  Peer Benchmarking
  DCF Valuation Model


In [ ]:
from google.colab import files
files.download("clean_financial_data.csv")
files.download("Financial_Due_Diligence_Report.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df = pd.read_csv("clean_financial_data.csv")
print(df[["Company","ROE","ROA"]].to_string())

                     Company    ROE    ROA
0                  HDFC Bank  14.02   1.75
1                  Axis Bank  13.63   1.56
2                 ICICI Bank  16.80   2.14
3                    Infosys  32.68  15.67
4                    HCLTech  23.19  12.87
5                        ITC   0.00   0.00
6              Eicher Motors   0.00   0.00
7   Dr. Reddy's Laboratories  16.10   7.96
8          Adani Ports & SEZ  19.41   0.00
9         Bharat Electronics   0.00   0.00
10                Coal India   0.00   0.00
11             Bharti Airtel  23.12   7.77
12                 HDFC Life  11.32   0.37
13        Hindustan Unilever   0.00   0.00
14                Bajaj Auto   0.00   0.00
15                     Cipla   0.00   0.00
16             Bajaj Finance   0.00   0.00
17                   Eternal   0.00   0.00
18              Asian Paints   0.00   0.00
19       Hindalco Industries   0.00   0.00
20         Adani Enterprises   0.00   0.00
21         Grasim Industries   0.00   0.00
22         

**Solving the multiple 0 problem in roe and roa**

In [ ]:
import pandas as pd

df = pd.read_csv("clean_financial_data.csv")

# manually filling ROE and ROA from screener.in for missing companies
manual_data = {
    "ITC":                  {"ROE": 28.4,  "ROA": 20.1},
    "Eicher Motors":        {"ROE": 24.8,  "ROA": 18.2},
    "Bharat Electronics":   {"ROE": 27.1,  "ROA": 16.4},
    "Coal India":           {"ROE": 52.3,  "ROA": 18.6},
    "Hindustan Unilever":   {"ROE": 19.8,  "ROA": 14.2},
    "Bajaj Auto":           {"ROE": 24.1,  "ROA": 16.8},
    "Cipla":                {"ROE": 14.2,  "ROA": 8.4},
    "Bajaj Finance":        {"ROE": 21.4,  "ROA": 3.8},
    "Eternal":              {"ROE": 8.2,   "ROA": 4.1},
    "Asian Paints":         {"ROE": 22.6,  "ROA": 14.8},
    "Hindalco Industries":  {"ROE": 11.2,  "ROA": 4.8},
    "Adani Enterprises":    {"ROE": 9.4,   "ROA": 2.8},
    "Grasim Industries":    {"ROE": 4.2,   "ROA": 1.8},
    "Apollo Hospitals":     {"ROE": 16.8,  "ROA": 6.2},
    "Bajaj Finserv":        {"ROE": 12.4,  "ROA": 2.1},
    "IndiGo":               {"ROE": 98.4,  "ROA": 4.2},
    "Adani Ports & SEZ":    {"ROE": 19.4,  "ROA": 7.2},
}

for company, values in manual_data.items():
    mask = df["Company"] == company
    if mask.any():
        df.loc[mask, "ROE"] = values["ROE"]
        df.loc[mask, "ROA"] = values["ROA"]
        print(f"Updated: {company}")

df.to_csv("clean_financial_data.csv", index=False)
print("\nDone. Verify:")
print(df[["Company","ROE","ROA"]].to_string())

Updated: ITC
Updated: Eicher Motors
Updated: Bharat Electronics
Updated: Coal India
Updated: Hindustan Unilever
Updated: Bajaj Auto
Updated: Cipla
Updated: Bajaj Finance
Updated: Eternal
Updated: Asian Paints
Updated: Hindalco Industries
Updated: Adani Enterprises
Updated: Grasim Industries
Updated: Apollo Hospitals
Updated: Bajaj Finserv
Updated: IndiGo
Updated: Adani Ports & SEZ

Done. Verify:
                     Company    ROE    ROA
0                  HDFC Bank  14.02   1.75
1                  Axis Bank  13.63   1.56
2                 ICICI Bank  16.80   2.14
3                    Infosys  32.68  15.67
4                    HCLTech  23.19  12.87
5                        ITC  28.40  20.10
6              Eicher Motors  24.80  18.20
7   Dr. Reddy's Laboratories  16.10   7.96
8          Adani Ports & SEZ  19.40   7.20
9         Bharat Electronics  27.10  16.40
10                Coal India  52.30  18.60
11             Bharti Airtel  23.12   7.77
12                 HDFC Life  11.32   0.37

In [ ]:
from google.colab import files
files.download("clean_financial_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('sql_analysis_results.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>